# Start a conversation for each newsletter edition

Creates one Foundry conversation per edition of the fictional Contoso Daily Brief, with the edition's
articles in the conversation and the edition id in its metadata.
[08-13-02](08-13-02-return-and-resume.ipynb) then finds and resumes those conversations from a fresh kernel.
Background: [`08-13-00-newsletter-conversations.md`](08-13-00-newsletter-conversations.md).

**Prerequisites:** `uv sync`, `az login`, and `ALPHA_FOUNDRY_PROJECT_ENDPOINT`, `ALPHA_FOUNDRY_CORE_CONNECTION`
and `CHAT_MODEL` in `.env` (set by the 05 labs).

## Step 1: Configuration

Load `.env` and connect to the Team Alpha project.

In [1]:
import json, os, subprocess
from pathlib import Path
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition

repo_root = Path(subprocess.run(
    'git rev-parse --show-toplevel', shell=True, capture_output=True, text=True
).stdout.strip())
load_dotenv(repo_root / '.env', override=True)

PROJECT_ENDPOINT = os.environ["ALPHA_FOUNDRY_PROJECT_ENDPOINT"]
MODEL            = f'{os.environ["ALPHA_FOUNDRY_CORE_CONNECTION"]}/{os.environ["CHAT_MODEL"]}'
AGENT_NAME       = "newsletter-agent"
EDITIONS_DIR     = repo_root / "assets" / "contoso-daily-brief-dataset" / "editions"

client = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=DefaultAzureCredential())

print(f"Project : {PROJECT_ENDPOINT}")
print(f"Agent   : {AGENT_NAME}")
print(f"Model   : {MODEL}")

Project : https://aif-spoke-alpha-c2676f.services.ai.azure.com/api/projects/project-alpha-c2676f
Agent   : newsletter-agent
Model   : core-alpha/gpt-5.4-mini


## Step 2: Create the agent

A prompt agent with no tools. It answers from the edition stored in the conversation.

In [2]:
agent = client.agents.create_version(
    agent_name=AGENT_NAME,
    definition=PromptAgentDefinition(
        model=MODEL,
        instructions=(
            "You answer questions about one edition of the Contoso Daily Brief newsletter. "
            "The edition is the first message of the conversation. Answer only from that edition, "
            "and give the id of each article you use in square brackets."
        ),
    ),
)
agent_client = client.get_openai_client(agent_name=AGENT_NAME)

print(f"{agent.name} version {agent.version}")

newsletter-agent version 1


## Step 3: Load the editions

Read the five editions from `assets/` and print each one as the text that goes into its conversation. Each article keeps its id.

In [3]:
editions = [json.loads(path.read_text(encoding="utf-8")) for path in sorted(EDITIONS_DIR.glob("ed-*.json"))]

def edition_text(edition):
    """An edition as plain text, one block per article, each starting with the article id."""
    articles = "\n\n".join(
        f'[{a["article_id"]}] {a["category"]}: {a["headline"]}\n{a["summary"]}' for a in edition["articles"])
    return (f'Contoso Daily Brief, edition {edition["edition_id"]}, published {edition["published"]}\n'
            f'{edition["title"]}\n\n{articles}')

for e in editions:
    print(edition_text(e))
    print("\n" + "-" * 100 + "\n")

Contoso Daily Brief, edition ed-001, published 2026-11-16
Northwind buys into a port, Relecloud heats homes

[ed-001-01] Business: Northwind Traders buys 40% of the Alderbrook container terminal
Northwind Traders will pay 310 million euros for a 40% stake in the operator of the Alderbrook container terminal. The food distributor says direct berth access will cut two days from deliveries to its northern warehouses. The deal is due to close in the first quarter of 2027, subject to competition review.

[ed-001-02] Technology: Relecloud's liquid-cooled server hall now heats 2,400 homes
Relecloud has connected its newest server hall to the local district heating network, sending waste heat to 2,400 homes. In a six-month pilot the liquid cooling system used 38% less electricity than the air-cooled hall next door. The company plans to convert three more halls in 2027.

[ed-001-03] Science: Trey Research finds a kelp forest recovering twice as fast as predicted
A Trey Research survey of 12 coa

## Step 4: Create one conversation per edition

Each conversation gets the edition id, date and title as metadata, and the edition text as its first item
(role `developer`). An edition that already has a conversation is skipped, so the notebook can be re-run.

In [4]:
def list_conversations():
    """The conversations on this agent's endpoint, one row each: `id` (the Foundry conversation id), `created_at`
    and `metadata`. The openai SDK has no list method, so this is a raw GET."""
    return agent_client.get("/conversations", cast_to=dict, options={"params": {"limit": 100}})["data"]

existing = {c["metadata"].get("edition_id"): c["id"] for c in list_conversations()}
started = []
for e in editions:
    if e["edition_id"] in existing:
        print(f'{e["edition_id"]}  exists   {existing[e["edition_id"]]}')
        continue
    conversation = agent_client.conversations.create(
        metadata={"edition_id": e["edition_id"], "edition_date": e["published"], "edition_title": e["title"]},
        items=[{"type": "message", "role": "developer", "content": edition_text(e)}],
    )
    started.append((e, conversation.id))
    print(f'{e["edition_id"]}  created  {conversation.id}')

print()
print("Metadata of the last conversation created:" if started else "No new conversations: every edition already has one from an earlier run.")
if started:
    print(json.dumps(agent_client.conversations.retrieve(started[-1][1]).metadata, indent=2))

ed-001  created  conv_0c6a5f3c0806f98f00enUp9LBbfDgBxXLLJBuF7TDBCaS0PBs2


ed-002  created  conv_0ba77b81f77ee14100luVsOsTDPPV4hHcRR6M7MRM0TBgC5NcI
ed-003  created  conv_01aee59eb655757f00dAVT4YtQDPZCYpEpcPA6P9HewjNGjTyE


ed-004  created  conv_0046fb51d8a3fd3900qfWksCLd3qyNSHBLFITshuVenfcAYKEq
ed-005  created  conv_01cd091aec7edaa000GaduHhx7OFwkDlyh2g76FI3bqCo2Occ3

Metadata of the last conversation created:


{
  "edition_id": "ed-005",
  "edition_date": "2026-11-20",
  "edition_title": "Blue Yonder sells rail and air on one ticket, Trey Research drills into old ice"
}


## Step 5: Ask a first question in each new conversation

The question names one article as the reader's pick. No reader is choosing here, so the code always takes the
edition's second article. [08-13-02](08-13-02-return-and-resume.ipynb) later asks the conversation which article
was picked, to show that the history was kept.

The call passes only the Foundry conversation id and the question. The agent reads the edition from the
conversation, and Foundry appends the question and the answer to it.

Only the conversations that Step 4 created in this run get the question. When the notebook is run again, every
edition already has its conversation, so the cell asks nothing and says so.

In [5]:
for e, foundry_conv_id in started:
    picked_article_id = e["articles"][1]["article_id"]
    response = agent_client.responses.create(
        conversation=foundry_conv_id,
        input=f"My pick from this edition is {picked_article_id}. In two sentences, why does it matter?",
    )
    print(f'{e["edition_id"]}  {foundry_conv_id}\n{response.output_text}\n')

if not started:
    print("Nothing to ask: Step 4 created no conversations in this run, because every edition already has one.\n"
          "08-13-02 reads those conversations back.")

ed-001  conv_0c6a5f3c0806f98f00enUp9LBbfDgBxXLLJBuF7TDBCaS0PBs2
It matters because Relecloud is showing that a data center can do more than compute: it can also supply useful heat to 2,400 homes, turning waste into a local energy benefit [ed-001-02]. It also cut electricity use by 38% versus an air-cooled hall in the pilot, suggesting a more efficient model that could scale to three more halls in 2027 [ed-001-02].



ed-002  conv_0ba77b81f77ee14100luVsOsTDPPV4hHcRR6M7MRM0TBgC5NcI
It matters because Fabrikam is testing sodium-ion vans in real fleets, which could make delivery vehicles cheaper to buy and quicker to recharge if the technology scales [ed-002-02]. With Wide World Importers as the first customer, it also signals early commercial interest beyond the lab, not just a prototype demo [ed-002-02].



ed-003  conv_01aee59eb655757f00dAVT4YtQDPZCYpEpcPA6P9HewjNGjTyE
It matters because Proseware has made on-device AI practical on a mid-range phone, which could let apps work faster and without a network connection, improving privacy and reliability [ed-003-02]. It also signals that small, efficient language models may soon be good enough for real consumer features, not just cloud-based demos [ed-003-02].



ed-004  conv_0046fb51d8a3fd3900qfWksCLd3qyNSHBLFITshuVenfcAYKEq
It matters because the recall affects 60,000 tumble dryers, so many households may have a safety issue in a widely sold product. The faulty door sensor could let the drum turn with the door open, creating a risk of injury, even though no injuries have been reported yet. [ed-004-02]



ed-005  conv_01cd091aec7edaa000GaduHhx7OFwkDlyh2g76FI3bqCo2Occ3
It matters because Consolidated Messenger is testing a practical way to speed up parcel delivery on campuses while reducing the need for human couriers [ed-005-02]. The trial also shows how delivery robots can fit into everyday life by letting students pick up packages with a phone code after low-speed, short-range transport [ed-005-02].



---

[Next: Return to a conversation and resume it →](08-13-02-return-and-resume.ipynb)